# 이미지 임베딩 모델 4종 정량 비교

같은 프레임·같은 질문으로 네 모델을 나란히 잽니다.

| 이름 | 모델 |
| --- | --- |
| `siglip2_base` | google/siglip2-base-patch16-224 |
| `siglip2_naflex` | google/siglip2-base-patch16-naflex (원본 비율 유지) |
| `siglip2_lora` | 위 base + 우리가 파인튜닝한 LoRA |
| `jina_clip_v2` | jinaai/jina-clip-v2 |

**검색 단위는 프레임 1장(1초 간격)입니다.** 질문을 주면 모든 영상의 모든 프레임에
순위를 매기고, 정답 영상의 정답 시각 ±TOL 안에 있는 프레임이 상위 k에 들었는지를 셉니다.
"그 초로 데려다준다"는 제품의 약속을 그대로 지표로 옮긴 것입니다.

텍스트 기준선(BGE-M3)과, 텍스트+이미지 RRF 융합도 같이 잽니다.

---
### 실행 순서
1. 셀 1~6: 설치·자막·프레임 (demo.ipynb 캐시를 그대로 재사용합니다)
2. **셀 7: 정답 세트 작성 ← 사람이 하는 유일한 작업**
3. 셀 8~10: 네 모델로 프레임 인코딩 (모델당 1회, Drive에 캐시)
4. 셀 11~13: 평가표 출력


In [ ]:
!pip install -q "transformers>=4.51" "FlagEmbedding>=1.2" peft accelerate einops timm
!pip install -q openai-whisper opencv-python-headless pandas
!apt-get install -qq ffmpeg

In [ ]:
import os, json, gc, re
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from PIL import Image
from IPython.display import display

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

# ── 경로 (demo.ipynb 와 동일하게 맞추세요) ──────────────────────────────
DRIVE_BASE   = '/content/drive/MyDrive'
VIDEO_DIR    = f'{DRIVE_BASE}/recipegen/video2'
ADAPTER_PATH = f'{DRIVE_BASE}/Graduate_project/model/siglip2_lora_qv_r16_best'
GOLD_PATH    = f'{DRIVE_BASE}/Graduate_project/eval/gold.json'     # 정답 세트가 여기 저장됩니다
CACHE_DIR    = f'{DRIVE_BASE}/Graduate_project/eval/emb_cache'     # 프레임 벡터 캐시

WORK_DIR   = '/content/video_demo'
FRAMES_DIR = f'{WORK_DIR}/frames'
SUBS_DIR   = f'{WORK_DIR}/subtitles'

BGE_MODEL_ID   = 'BAAI/bge-m3'
FRAME_INTERVAL = 1.0
MAX_FRAMES     = 120

# ── 평가 기준 ────────────────────────────────────────────────────────
TOL_SECONDS = 2.5    # 정답 시각 ±2.5초 안의 프레임이면 맞은 것으로 봅니다
TOP_KS      = (1, 5, 10)

for d in [WORK_DIR, FRAMES_DIR, SUBS_DIR]:
    os.makedirs(d, exist_ok=True)
print('설정 완료')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

os.makedirs(os.path.dirname(GOLD_PATH), exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

video_paths = []
for ext in ['*.mp4', '*.avi', '*.mov', '*.mkv', '*.MP4']:
    video_paths.extend(Path(VIDEO_DIR).glob(ext))
video_paths = sorted(set(video_paths))
if not video_paths:
    raise FileNotFoundError(f'영상 없음: {VIDEO_DIR}')

print(f'영상 {len(video_paths)}개')
for p in video_paths:
    print(' ', p.name)

## 1. 자막 (Whisper) — demo.ipynb 캐시를 그대로 씁니다

In [ ]:
import whisper as openai_whisper

subtitles = {}
todo = [vp for vp in video_paths if not os.path.exists(f'{SUBS_DIR}/{vp.stem}.json')]

if todo:
    w_model = openai_whisper.load_model('base', device=device)
    for vp in todo:
        vid = vp.stem
        print(f'자막 추출: {vid} ...', end=' ')
        try:
            r = w_model.transcribe(str(vp), language='ko', verbose=False)
            segs = [{'start': s['start'], 'end': s['end'], 'text': s['text'].strip()}
                    for s in r['segments'] if s.get('text', '').strip()]
        except Exception as e:
            print(f'실패({e})'); segs = []
        with open(f'{SUBS_DIR}/{vid}.json', 'w', encoding='utf-8') as f:
            json.dump(segs, f, ensure_ascii=False, indent=2)
        print(f'{len(segs)}개')
    del w_model; gc.collect(); torch.cuda.empty_cache()

for vp in video_paths:
    with open(f'{SUBS_DIR}/{vp.stem}.json', encoding='utf-8') as f:
        subtitles[vp.stem] = json.load(f)
print('\n자막 준비 완료:', {k: len(v) for k, v in subtitles.items()})

## 2. 프레임 추출 (1초 간격)

In [ ]:
import cv2

def extract_frames(video_path, vid):
    frame_dir  = f'{FRAMES_DIR}/{vid}'
    cache_path = f'{frame_dir}/index.json'
    if os.path.exists(cache_path):
        with open(cache_path) as f:
            return json.load(f)

    os.makedirs(frame_dir, exist_ok=True)
    cap      = cv2.VideoCapture(str(video_path))
    fps      = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total    = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total / fps
    interval = max(FRAME_INTERVAL, duration / MAX_FRAMES)

    frames = []
    for ts in np.arange(0, duration - 0.5, interval):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(ts * fps))
        ok, frame = cap.read()
        if not ok:
            continue
        path = f'{frame_dir}/{ts:.2f}.jpg'
        cv2.imwrite(path, frame)
        frames.append({'timestamp': float(ts), 'path': path})
    cap.release()
    with open(cache_path, 'w') as f:
        json.dump(frames, f)
    return frames

keyframes = {vp.stem: extract_frames(vp, vp.stem) for vp in video_paths}

# ── 평가 단위: 프레임 한 장 = 후보 하나 ────────────────────────────────
def subtitle_at(vid, ts):
    for s in subtitles.get(vid, []):
        if s['start'] <= ts <= s['end']:
            return s['text']
    return ''

ITEMS = []   # 전 영상의 모든 프레임을 한 줄로 폅니다
for vp in video_paths:
    vid = vp.stem
    for f in keyframes[vid]:
        ITEMS.append({'video_id': vid, 'time': f['timestamp'],
                      'path': f['path'], 'text': subtitle_at(vid, f['timestamp'])})

ITEM_VIDEO = np.array([it['video_id'] for it in ITEMS])
ITEM_TIME  = np.array([it['time'] for it in ITEMS], dtype=np.float32)
ITEM_PATHS = [it['path'] for it in ITEMS]
ITEM_TEXTS = [it['text'] or ' ' for it in ITEMS]
# 자막이 없는 프레임은 텍스트 근거가 아예 없습니다. 텍스트 점수에서 빼둡니다.
HAS_TEXT   = np.array([bool(it['text']) for it in ITEMS])

print(f'후보 프레임 {len(ITEMS)}개 / 영상 {len(video_paths)}개')
print(f'자막이 붙은 프레임 {sum(1 for it in ITEMS if it["text"])}개')

## 3. 정답 세트 ← **여기만 사람이 합니다**

아래 셀을 먼저 돌려 영상별 자막을 시각과 함께 출력한 뒤, 그걸 보면서 질문을 만드세요.

**질문 유형을 꼭 나눠 적으세요.** 나중에 "이미지 모델은 visual 질의에서만 차이가 난다"
같은 결론이 이 필드에서 나옵니다.

- `visual` — 화면을 봐야 답이 나오는 것 (`파 써는 장면`, `빨간 냄비`)
- `speech` — 말에서 답이 나오는 것 (`감자 언제 넣어?`)
- `lexical` — 숫자·고유명사 (`간장 2큰술`)

영상 10개면 **영상당 4~5개, 총 40~50개**를 목표로 하세요. 30개 미만이면 조건 간
차이가 우연인지 아닌지 구분이 안 됩니다.

In [ ]:
# 라벨링 보조: 자막을 시각과 함께 찍어 봅니다
for vp in video_paths:
    vid = vp.stem
    print(f'\n{"─"*60}\n[{vid}]  프레임 {len(keyframes[vid])}장')
    for s in subtitles.get(vid, []):
        print(f'  {s["start"]:6.1f}s ~ {s["end"]:6.1f}s | {s["text"]}')

In [ ]:
# ── 정답 세트 ─────────────────────────────────────────────────────────
# 아래 예시를 지우고 직접 채우세요. gold_time = 답이 되는 장면의 시작 시각(초).
GOLD = [
    {"query": "파 써는 장면",            "video_id": "short_001", "gold_time": 31.2, "type": "visual"},
    {"query": "감자 넣는 장면",          "video_id": "short_001", "gold_time": 24.3, "type": "visual"},
    {"query": "감자탕에 뭐부터 넣어?",   "video_id": "short_001", "gold_time":  7.7, "type": "speech"},
    {"query": "사골곰탕",                "video_id": "short_001", "gold_time": 14.0, "type": "lexical"},
    # ... 여기에 계속 추가 ...
]

with open(GOLD_PATH, 'w', encoding='utf-8') as f:
    json.dump(GOLD, f, ensure_ascii=False, indent=2)
print(f'{len(GOLD)}개 저장 → {GOLD_PATH}')

In [ ]:
# ── 정답 세트 검증 ────────────────────────────────────────────────────
with open(GOLD_PATH, encoding='utf-8') as f:
    GOLD = json.load(f)

valid_vids = {vp.stem for vp in video_paths}
problems = []
for i, g in enumerate(GOLD):
    if g['video_id'] not in valid_vids:
        problems.append(f'[{i}] 없는 영상: {g["video_id"]}')
        continue
    mask = ITEM_VIDEO == g['video_id']
    if not (np.abs(ITEM_TIME[mask] - g['gold_time']) <= TOL_SECONDS).any():
        problems.append(f'[{i}] {g["video_id"]} {g["gold_time"]}s 근처에 프레임이 없습니다')
    if g.get('type') not in ('visual', 'speech', 'lexical'):
        problems.append(f'[{i}] type 이 visual/speech/lexical 중 하나여야 합니다')

print(f'질의 {len(GOLD)}개')
print(pd.Series([g.get("type") for g in GOLD]).value_counts().to_string())
if problems:
    print('\n⚠ 문제:'); [print(' ', p) for p in problems]
else:
    print('\n✅ 정답 세트 이상 없음')
if len(GOLD) < 30:
    print(f'\n⚠ 질의가 {len(GOLD)}개뿐입니다. 30개 미만이면 조건 간 차이를 신뢰하기 어렵습니다.')

## 4. 모델 4종

**중요 — SigLIP 계열의 텍스트 입력은 반드시 `padding="max_length"` 여야 합니다.**
SigLIP 은 64토큰 고정 길이로 학습돼서, `padding=True`(가변 길이)로 넣으면 텍스트 쪽
품질이 눈에 띄게 떨어집니다. 기존 `demo.ipynb` 21번 셀이 `padding=True` 로 되어 있어
SigLIP 검색 결과가 실제보다 나쁘게 나왔을 가능성이 있습니다. 여기서는 고쳐 두었습니다.

In [ ]:
from transformers import AutoModel, AutoProcessor

def l2n(v):
    v = np.asarray(v, dtype=np.float32)
    n = np.linalg.norm(v, axis=-1, keepdims=True)
    return v / np.clip(n, 1e-12, None)


class SiglipEncoder:
    """SigLIP2 base / NaFlex / LoRA 를 모두 같은 인터페이스로 감쌉니다."""
    def __init__(self, model_id, adapter=None):
        dtype = torch.float16 if device == 'cuda' else torch.float32
        self.proc  = AutoProcessor.from_pretrained(model_id)
        m = AutoModel.from_pretrained(model_id, torch_dtype=dtype).to(device)
        if adapter:
            from peft import PeftModel
            m = PeftModel.from_pretrained(m, adapter)
        self.model = m.eval()

    def _img_features(self, batch):
        # PeftModel 로 감싸도 get_image_features 는 그대로 닿습니다. 혹시 못 닿으면
        # vision_model 의 pooler 로 대체합니다(SigLIP 은 둘이 같은 값입니다).
        try:
            return self.model.get_image_features(**batch)
        except AttributeError:
            return self.model.vision_model(**batch).pooler_output

    @torch.no_grad()
    def embed_images(self, paths, bs=16):
        out = []
        for i in range(0, len(paths), bs):
            imgs  = [Image.open(p).convert('RGB') for p in paths[i:i+bs]]
            batch = self.proc(images=imgs, return_tensors='pt').to(device)
            out.append(self._img_features(batch).float().cpu().numpy())
        return l2n(np.vstack(out))

    @torch.no_grad()
    def embed_texts(self, texts, bs=32):
        out = []
        for i in range(0, len(texts), bs):
            # padding='max_length' 가 핵심입니다. 학습 조건과 맞춰야 합니다.
            batch = self.proc(text=list(texts[i:i+bs]), padding='max_length',
                              max_length=64, truncation=True, return_tensors='pt').to(device)
            batch = {k: v for k, v in batch.items() if 'pixel' not in k and 'spatial' not in k}
            try:
                f = self.model.get_text_features(**batch)
            except AttributeError:
                f = self.model.text_model(**batch).pooler_output
            out.append(f.float().cpu().numpy())
        return l2n(np.vstack(out))


class JinaEncoder:
    def __init__(self, model_id='jinaai/jina-clip-v2'):
        dtype = torch.float16 if device == 'cuda' else torch.float32
        self.model = AutoModel.from_pretrained(
            model_id, trust_remote_code=True, torch_dtype=dtype).to(device).eval()

    def embed_images(self, paths, bs=16):
        return l2n(self.model.encode_image([str(p) for p in paths], batch_size=bs))

    def embed_texts(self, texts, bs=32):
        try:   # jina-clip-v2 는 질의용 task 어댑터를 갖고 있습니다
            v = self.model.encode_text(list(texts), batch_size=bs, task='retrieval.query')
        except TypeError:
            v = self.model.encode_text(list(texts), batch_size=bs)
        return l2n(v)


MODELS = {
    'siglip2_base':   lambda: SiglipEncoder('google/siglip2-base-patch16-224'),
    'siglip2_naflex': lambda: SiglipEncoder('google/siglip2-base-patch16-naflex'),
    'siglip2_lora':   lambda: SiglipEncoder('google/siglip2-base-patch16-224', adapter=ADAPTER_PATH),
    'jina_clip_v2':   lambda: JinaEncoder(),
}
print('등록된 모델:', list(MODELS))

## 5. 프레임 인코딩 (모델당 1회, Drive 캐시)

모델을 하나씩 올렸다 내리므로 메모리가 겹치지 않습니다. 한 모델이라도 로드에
실패하면 그 모델만 건너뛰고 나머지는 계속 평가합니다.

In [ ]:
QUERIES = [g['query'] for g in GOLD]

img_vecs = {}    # model -> (프레임수, dim)
qry_vecs = {}    # model -> (질의수, dim)
failed   = {}

for name, factory in MODELS.items():
    cache = f'{CACHE_DIR}/{name}.npz'
    if os.path.exists(cache):
        z = np.load(cache, allow_pickle=True)
        if len(z['images']) == len(ITEM_PATHS) and list(z['queries_src']) == QUERIES:
            img_vecs[name], qry_vecs[name] = z['images'], z['queries']
            print(f'[캐시] {name}: 이미지 {z["images"].shape}, 질의 {z["queries"].shape}')
            continue
        print(f'[캐시 무효] {name}: 프레임/질의가 바뀌었습니다. 다시 인코딩합니다.')

    print(f'\n▶ {name} 로드 중...')
    enc = None
    try:
        enc = factory()
        img_vecs[name] = enc.embed_images(ITEM_PATHS)
        qry_vecs[name] = enc.embed_texts(QUERIES)
        np.savez(cache, images=img_vecs[name], queries=qry_vecs[name],
                 queries_src=np.array(QUERIES, dtype=object))
        print(f'  완료: 이미지 {img_vecs[name].shape}, 질의 {qry_vecs[name].shape}')
    except Exception as e:
        failed[name] = f'{type(e).__name__}: {e}'
        print(f'  ❌ 실패 — {failed[name]}')
    finally:
        enc = None
        gc.collect()
        if device == 'cuda':
            torch.cuda.empty_cache()

print('\n사용 가능:', list(img_vecs))
if failed:
    print('실패:', failed)

In [ ]:
# ── 텍스트 기준선: BGE-M3 (프레임에 붙은 자막을 인코딩) ────────────────
from FlagEmbedding import BGEM3FlagModel

bge_cache = f'{CACHE_DIR}/bge_m3.npz'
bge_item = bge_query = None

if os.path.exists(bge_cache):
    z = np.load(bge_cache, allow_pickle=True)
    if len(z['images']) == len(ITEM_TEXTS) and list(z['queries_src']) == QUERIES:
        bge_item, bge_query = z['images'], z['queries']
        print('[캐시] bge_m3')

if bge_item is None:
    bge = BGEM3FlagModel(BGE_MODEL_ID, use_fp16=(device == 'cuda'))
    bge_item  = l2n(bge.encode(ITEM_TEXTS, batch_size=64, max_length=256)['dense_vecs'])
    bge_query = l2n(bge.encode(QUERIES,    batch_size=32, max_length=256)['dense_vecs'])
    np.savez(bge_cache, images=bge_item, queries=bge_query,
             queries_src=np.array(QUERIES, dtype=object))
    del bge
    gc.collect()
    if device == 'cuda':
        torch.cuda.empty_cache()
    print('BGE-M3 인코딩 완료')

print('자막 벡터', bge_item.shape, '질의 벡터', bge_query.shape)

## 6. 평가

- **Hit@k** — 상위 k개 안에 정답 영상의 정답 시각 ±TOL 프레임이 있는가
- **MRR** — 첫 정답 프레임 등수의 역수, 평균
- **영상 R@1** — 시각은 틀려도 1등 프레임이 정답 영상에서 나왔는가
- **시각오차** — 영상을 맞춘 경우 1등 프레임과 정답 시각의 차이(중앙값)

융합은 RRF(Reciprocal Rank Fusion)입니다. 두 모델의 점수 크기가 달라도
등수만 쓰므로 안전합니다.

In [ ]:
def gold_mask(g):
    """이 질의의 정답으로 인정할 프레임들."""
    return (ITEM_VIDEO == g['video_id']) & (np.abs(ITEM_TIME - g['gold_time']) <= TOL_SECONDS)


def ranking_from_scores(scores):
    return np.argsort(-scores, kind='stable')


def rrf_rank(rank_lists, k=60):
    n = len(ITEMS)
    fused = np.zeros(n, dtype=np.float64)
    for order in rank_lists:
        pos = np.empty(n, dtype=np.int64)
        pos[order] = np.arange(n)
        fused += 1.0 / (k + pos + 1)
    return np.argsort(-fused, kind='stable')


def evaluate(rank_fn, subset=None):
    """rank_fn(qi) -> 프레임 인덱스 순위 배열."""
    rows = []
    for qi, g in enumerate(GOLD):
        if subset and g.get('type') != subset:
            continue
        order = rank_fn(qi)
        good  = gold_mask(g)
        hits  = np.where(good[order])[0]          # 정답 프레임들의 등수
        first = int(hits[0]) if len(hits) else None
        top1  = order[0]
        rows.append({
            **{f'hit@{k}': (first is not None and first < k) for k in TOP_KS},
            'rr':          (1.0 / (first + 1)) if first is not None else 0.0,
            'video_r@1':   ITEM_VIDEO[top1] == g['video_id'],
            'time_err':    abs(float(ITEM_TIME[top1]) - g['gold_time'])
                           if ITEM_VIDEO[top1] == g['video_id'] else np.nan,
        })
    if not rows:
        return None
    df = pd.DataFrame(rows)
    out = {f'Hit@{k}': df[f'hit@{k}'].mean() for k in TOP_KS}
    out['MRR']      = df['rr'].mean()
    out['영상R@1']  = df['video_r@1'].mean()
    out['시각오차'] = df['time_err'].median()
    out['n']        = len(df)
    return out


# ── 조건 정의 ────────────────────────────────────────────────────────
def text_rank(qi):
    s = bge_item @ bge_query[qi]
    s = np.where(HAS_TEXT, s, -1e9)   # 자막 없는 프레임은 텍스트 근거가 없습니다
    return ranking_from_scores(s)

def image_rank_factory(name):
    def f(qi):
        return ranking_from_scores(img_vecs[name] @ qry_vecs[name][qi])
    return f

def fused_rank_factory(name):
    def f(qi):
        return rrf_rank([text_rank(qi), image_rank_factory(name)(qi)])
    return f

CONDITIONS = {'텍스트만 (BGE-M3)': text_rank}
for name in img_vecs:
    CONDITIONS[f'이미지만 · {name}'] = image_rank_factory(name)
for name in img_vecs:
    CONDITIONS[f'BGE + {name}'] = fused_rank_factory(name)

print(f'조건 {len(CONDITIONS)}개, 질의 {len(GOLD)}개, 후보 프레임 {len(ITEMS)}개')

In [ ]:
# ── 전체 결과 ────────────────────────────────────────────────────────
rows = {}
for label, fn in CONDITIONS.items():
    rows[label] = evaluate(fn)

table = pd.DataFrame(rows).T
table = table[[f'Hit@{k}' for k in TOP_KS] + ['MRR', '영상R@1', '시각오차', 'n']]
display(table.style.format({**{f'Hit@{k}': '{:.3f}' for k in TOP_KS},
                            'MRR': '{:.3f}', '영상R@1': '{:.3f}',
                            '시각오차': '{:.1f}s', 'n': '{:.0f}'})
             .background_gradient(subset=[f'Hit@{k}' for k in TOP_KS] + ['MRR'], cmap='Greens'))

In [ ]:
# ── 질의 유형별 (이미지 모델의 진짜 차이는 여기서 납니다) ──────────────
for qtype in ['visual', 'speech', 'lexical']:
    rows = {}
    for label, fn in CONDITIONS.items():
        r = evaluate(fn, subset=qtype)
        if r:
            rows[label] = r
    if not rows:
        continue
    print(f'\n{"="*70}\n[{qtype} 질의]')
    t = pd.DataFrame(rows).T[[f'Hit@{k}' for k in TOP_KS] + ['MRR', '시각오차', 'n']]
    display(t.style.format({**{f'Hit@{k}': '{:.3f}' for k in TOP_KS},
                            'MRR': '{:.3f}', '시각오차': '{:.1f}s', 'n': '{:.0f}'}))

## 7. 실패 사례 보기

숫자만으로는 "왜 졌는지" 를 발표에서 설명할 수 없습니다. 틀린 질의를 직접 보세요.

In [ ]:
def show_failures(label, limit=5):
    fn = CONDITIONS[label]
    print(f'\n[{label}] 1등이 틀린 질의\n' + '='*70)
    shown = 0
    for qi, g in enumerate(GOLD):
        order = fn(qi)
        if gold_mask(g)[order[0]]:
            continue
        top = order[0]
        print(f'\nQ: {g["query"]}  ({g.get("type")})')
        print(f'   정답: {g["video_id"]} @ {g["gold_time"]:.1f}s')
        print(f'   1등 : {ITEM_VIDEO[top]} @ {ITEM_TIME[top]:.1f}s | 자막: {ITEMS[top]["text"][:40]}')
        try:
            display(Image.open(ITEM_PATHS[top]).resize((180, 320)))
        except Exception:
            pass
        shown += 1
        if shown >= limit:
            break
    if shown == 0:
        print('  (전부 맞췄습니다)')

for label in list(CONDITIONS)[:2]:
    show_failures(label, limit=3)

In [ ]:
# ── 발표용 요약 문장 자동 생성 ────────────────────────────────────────
best_img  = max(img_vecs, key=lambda n: evaluate(image_rank_factory(n))['MRR'])
base_mrr  = evaluate(text_rank)['MRR']
fused_mrr = evaluate(fused_rank_factory(best_img))['MRR']

vis = {n: evaluate(image_rank_factory(n), subset='visual') for n in img_vecs}
vis = {k: v for k, v in vis.items() if v}

print(f'질의 {len(GOLD)}개, 후보 프레임 {len(ITEMS)}개, 허용 오차 ±{TOL_SECONDS}초\n')
print(f'· 이미지 모델 4종 중 MRR 최고: {best_img}')
if vis:
    print('· visual 질의 Hit@5:')
    for n, r in sorted(vis.items(), key=lambda x: -x[1]["Hit@5"]):
        print(f'    {n:16} {r["Hit@5"]:.3f}')
if 'siglip2_lora' in vis and 'siglip2_base' in vis:
    d = vis['siglip2_lora']['Hit@5'] - vis['siglip2_base']['Hit@5']
    print(f'· 파인튜닝 효과(visual Hit@5): {d:+.3f}')
print(f'· 텍스트만 MRR {base_mrr:.3f} → BGE+{best_img} 융합 {fused_mrr:.3f} ({fused_mrr-base_mrr:+.3f})')